# 1. Async Context Managers (async with)

In regular Python, you use with open(...) as f: to ensure a file is safely closed even if an error occurs. This uses the __enter__ and __exit__ magic methods.

However, establishing a database connection or opening a network session (like with aiohttp) requires I/O operations (network handshakes). You cannot put an await inside a synchronous __enter__ method.

The Solution: Async context managers use __aenter__ and __aexit__. This allows you to safely acquire and release resources that require network or disk I/O.

# 2. Async Iterators and Generators (async for)
Imagine you are fetching 10,000 records from a database. Loading them all into memory at once is a bad idea. Normally, you'd use a generator (yield).

But what if fetching the next chunk of data requires an I/O call? You need an async generator. By using yield inside an async def function, you create a stream of data where each chunk can be awaited. You consume this stream using async for.

# 3. The Golden Rule & Bridging Sync/Async (asyncio.to_thread())
The Golden Rule of Async: Never block the event loop.
Because the event loop runs in a single thread, if you execute a blocking synchronous function (like time.sleep(), a heavy CPU calculation like hashing a password, or a synchronous request via requests.get()), every other task in your application completely freezes.

The Escape Hatch: asyncio.to_thread() (introduced in Python 3.9).
This function takes a blocking synchronous function and offloads it to a background thread. It returns an awaitable coroutine. This keeps your event loop free to handle other async requests while the heavy CPU work or legacy sync I/O finishes in the background thread.

In [2]:
import asyncio
import time

# ==========================================
# 1. Async Context Manager (Resource Safety)
# ==========================================
class AsyncDatabaseConnection:
    def __init__(self, db_url):
        self.db_url = db_url

    async def __aenter__(self):
        print(f"[DB] Connecting to {self.db_url} (Requires Network I/O)...")
        await asyncio.sleep(1) # Simulating network handshake
        print("[DB] Connected!")
        return self

    async def __aexit__(self, exc_type, exc, tb):
        print("[DB] Closing connection safely (Requires Network I/O)...")
        await asyncio.sleep(0.5) # Simulating graceful teardown
        print("[DB] Disconnected.")

# ==========================================
# 2. Async Generator (Data Streaming)
# ==========================================
async def fetch_paginated_users(pages: int):
    """Simulates fetching pages of users from a DB, one page at a time."""
    for page in range(1, pages + 1):
        print(f"[API] Fetching page {page} from database...")
        await asyncio.sleep(1) # Simulating DB query time
        yield [f"User_{page}_A", f"User_{page}_B"] # Yielding the chunk

# ==========================================
# 3. Blocking Code (The Threat to the Loop)
# ==========================================
def cpu_bound_password_hash(password: str):
    """A synchronous, CPU-heavy operation (e.g., bcrypt hashing)."""
    print(f"[CPU] Thread started hashing for '{password}'...")
    time.sleep(2) # BLOCKING CALL! If run directly, it freezes the async loop.
    print(f"[CPU] Hashing complete for '{password}'")
    return f"hashed_{password}_123"

# ==========================================
# The Orchestrator
# ==========================================
async def main():
    start_time = time.time()
    # Using 'async with' to manage the DB lifecycle
    async with AsyncDatabaseConnection("mongodb://localhost:27017") as db:
        # Using 'async for' to consume streamed data without blowing up memory
        print("\n--- Starting Data Stream ---")
        async for user_batch in fetch_paginated_users(pages=3):
            print(f"[App] Processing batch: {user_batch}")
            
        print("\n--- Bridging Sync and Async ---")
        # We need to hash a password, but we don't want to freeze the event loop.
        # We also want to do a lightweight async task at the same time to prove 
        # the loop isn't blocked.
        
        async def lightweight_task():
            for i in range(3):
                print(f"[Async] Loop is still running freely... ({i+1}/3)")
                await asyncio.sleep(0.7)
                
        # Run the CPU-bound task in a separate thread, but 'await' it here.
        # Run the lightweight task concurrently using gather.
        hash_task = asyncio.to_thread(cpu_bound_password_hash, "my_secure_pwd")
        light_task = lightweight_task()
        
        results = await asyncio.gather(hash_task, light_task)
        print(f"[App] Final Hash: {results[0]}")
        
    print(f"\nTotal Execution Time: {time.time() - start_time:.2f}s")

if __name__ == "__main__":
    # asyncio.run(main())
    await main()

[DB] Connecting to mongodb://localhost:27017 (Requires Network I/O)...
[DB] Connected!

--- Starting Data Stream ---
[API] Fetching page 1 from database...
[App] Processing batch: ['User_1_A', 'User_1_B']
[API] Fetching page 2 from database...
[App] Processing batch: ['User_2_A', 'User_2_B']
[API] Fetching page 3 from database...
[App] Processing batch: ['User_3_A', 'User_3_B']

--- Bridging Sync and Async ---
[CPU] Thread started hashing for 'my_secure_pwd'...
[Async] Loop is still running freely... (1/3)
[Async] Loop is still running freely... (2/3)
[Async] Loop is still running freely... (3/3)
[CPU] Hashing complete for 'my_secure_pwd'
[App] Final Hash: hashed_my_secure_pwd_123
[DB] Closing connection safely (Requires Network I/O)...
[DB] Disconnected.

Total Execution Time: 6.66s
